# Train and Register a Titanic Survival Model with MLflow

This notebook demonstrates a complete machine learning workflow:
1.  **Load Data**: Fetch the Titanic dataset.
2.  **Preprocess Data**: Create a pipeline to handle missing values, scale numeric features, and one-hot encode categorical features.
3.  **Train Model**: Use a Logistic Regression model.
4.  **Track with MLflow**: Log parameters, metrics, and the trained model to an MLflow tracking server.
5.  **Register Model**: Register the logged model in the MLflow Model Registry.

## 1. Imports

First, we import all the necessary libraries.

In [1]:
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

## 2. MLflow Configuration

Next, we set up the connection to the MLflow tracking server. This notebook uses a `.env` file in the same directory to store the `MLFLOW_TRACKING_URI`. 

Your `.env` file should contain a single line:
```
MLFLOW_TRACKING_URI=[http://127.0.0.1:5000](http://127.0.0.1:5000)
```
Replace the URI with the actual address of your MLflow server.

In [2]:
mlflow_tracking_uri = "http://localhost:5000"
mlflow.set_tracking_uri(mlflow_tracking_uri)

## 3. Load Data

We'll load the Titanic dataset directly from a URL. For offline use, you can download the CSV and load it locally.

In [3]:
df = pd.read_csv('data/data.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 4. Feature Engineering and Selection

We select the features we'll use for training and define our target variable. We also drop rows where the `Embarked` column is missing, as it's a key categorical feature.

In [4]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

# Drop rows with missing 'Embarked' values for simplicity
df_clean = df[features + [target]].dropna(subset=['Embarked']).copy()

X = df_clean[features]
y = df_clean[target]

print("Features and target defined.")
X.info()

Features and target defined.
<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    889 non-null    int64  
 1   Sex       889 non-null    object 
 2   Age       712 non-null    float64
 3   SibSp     889 non-null    int64  
 4   Parch     889 non-null    int64  
 5   Fare      889 non-null    float64
 6   Embarked  889 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 55.6+ KB


## 5. Preprocessing Pipeline

We create a preprocessing pipeline using `ColumnTransformer` to apply different transformations to numeric and categorical columns.

- **Numeric Features**: Impute missing values (e.g., in `Age`) with the median, then scale them using `StandardScaler`.
- **Categorical Features**: Impute missing values with the most frequent value, then convert them to a numeric format using `OneHotEncoder`.

In [5]:
# Define numeric and categorical feature groups
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Embarked', 'Sex', 'Pclass']

# Create a pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

# Create a pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

## 6. Define the Full Model Pipeline

Now we combine the preprocessor with our classifier (`LogisticRegression`) into a single `Pipeline`. This ensures that the same preprocessing steps are applied consistently during training and prediction.

In [6]:
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', LogisticRegression())])

## 7. Split Data into Training and Test Sets

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (711, 7)
Test set shape: (178, 7)


## 8. Train Model and Log with MLflow

This is the core of the workflow. We start an MLflow run to track our experiment.

Inside the `mlflow.start_run()` block:
1.  We fit the model pipeline on the training data.
2.  We evaluate the model on the test set and log the `accuracy` metric.
3.  We log the entire Scikit-learn pipeline as a model artifact and register it with a specific name in the Model Registry.

In [8]:
model_name = "TitanicSurvival"
mlflow.set_experiment(model_name)

print(f"Starting MLflow run under experiment '{model_name}'...")

with mlflow.start_run() as run:
    # Train the model
    model_pipeline.fit(X_train, y_train)

    # Evaluate and log metrics
    accuracy = model_pipeline.score(X_test, y_test)
    print(f"Model Accuracy: {accuracy:.4f}")
    mlflow.log_metric("accuracy", accuracy)

    # Log the model with an artifact path and register it
    print(f"Logging and registering model as '{model_name}'...")
    mlflow.sklearn.log_model(
        sk_model=model_pipeline,
        artifact_path="model",
        registered_model_name=model_name
    )

print("\n✅ Model trained and registered in MLflow successfully!")
print(f"Run ID: {run.info.run_id}")
print(f"Navigate to {mlflow.get_tracking_uri()} to see your experiment.")

Starting MLflow run under experiment 'TitanicSurvival'...


2025/07/27 13:23:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model Accuracy: 0.7865
Logging and registering model as 'TitanicSurvival'...


2025/07/27 13:23:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'TitanicSurvival' already exists. Creating a new version of this model...
2025/07/27 13:23:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: TitanicSurvival, version 3


🏃 View run blushing-bear-431 at: http://localhost:5000/#/experiments/942758532782988433/runs/3e4e4eb39df049caae9eb872a8c7d411
🧪 View experiment at: http://localhost:5000/#/experiments/942758532782988433

✅ Model trained and registered in MLflow successfully!
Run ID: 3e4e4eb39df049caae9eb872a8c7d411
Navigate to http://localhost:5000 to see your experiment.


Created version '3' of model 'TitanicSurvival'.
